# Lecture 4.2 — Read-Only Agents vs Auto-Edit Agents

**Section 4 — Permissions & Safety**  
**Course: Claude Agent SDK: Build AI Agents in Python**

---

In this notebook we build **three agent configurations** and run them all on the same task.

| Configuration | allowed_tools | permission_mode | Can modify files? |
|---|---|---|---|
| Read-Only Agent | Read, Glob, Grep | dontAsk | ❌ |
| Auto-Edit Combo 1 | Read, **Write, Edit**, Glob, Grep | dontAsk | ✅ |
| Auto-Edit Combo 2 | Read, Glob, Grep | **acceptEdits** | ✅ |

**Key insight:** Combo 1 and Combo 2 produce identical output through different gate paths.

In [ ]:
# Install the Claude Agent SDK

# We pin the Claude Agent SDK to a specific version to ensure all examples
# in this notebook run exactly as shown in the course. If you encounter any
# issues or want to experiment with newer features, you can install the latest
# version by removing the version pin (replace 'claude-agent-sdk==0.2.93' with just
# 'claude-agent-sdk'). Note that newer versions may behave differently from
# what is demonstrated in the videos. We will update the notebooks periodically
# to keep up with new releases.

!pip install claude-agent-sdk==0.2.93 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.6/74.6 MB 11.5 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

## Model Configuration

We pin the model name as a variable so you can change it in one place.  
For the latest available models: https://platform.claude.com/docs/en/about-claude/models/overview

In [ ]:
# Model configuration
# Change this to use a different Claude model
# For the latest available models visit:
# https://platform.claude.com/docs/en/about-claude/models/overview
MODEL_NAME = "claude-haiku-4-5"

## The Sample Project

We create a small Python project with **four files and nine functions** — every function is deliberately missing a docstring.

All three agents will be asked to find those missing docstrings. Only the Auto-Edit Agents will be allowed to add them.

In [ ]:
import os

os.makedirs("/content/docstring_project/src", exist_ok=True)
os.makedirs("/content/docstring_project/tests", exist_ok=True)

with open("/content/docstring_project/src/auth.py", "w") as f:
    f.write("# Authentication module\n\n")
    f.write("def login(user, password):\n")
    f.write("    return user == 'admin' and password == 'secret'\n\n")
    f.write("def logout(user):\n")
    f.write("    return True\n\n")
    f.write("def reset_password(user, new_password):\n")
    f.write("    return True\n")

with open("/content/docstring_project/src/utils.py", "w") as f:
    f.write("# Utility functions\n\n")
    f.write("def format_date(date):\n")
    f.write("    return date.strftime('%Y-%m-%d')\n\n")
    f.write("def format_currency(amount):\n")
    f.write("    return f'${amount:.2f}'\n\n")
    f.write("def validate_email(email):\n")
    f.write("    return '@' in email\n")

with open("/content/docstring_project/src/payments.py", "w") as f:
    f.write("# Payments module\n\n")
    f.write("def process_payment(amount, card):\n")
    f.write("    return {'status': 'success', 'amount': amount}\n\n")
    f.write("def refund_payment(transaction_id):\n")
    f.write("    return {'status': 'refunded', 'id': transaction_id}\n")

with open("/content/docstring_project/tests/test_auth.py", "w") as f:
    f.write("# Tests for auth module\n\n")
    f.write("def test_login():\n")
    f.write("    assert login('admin', 'secret') == True\n\n")
    f.write("def test_logout():\n")
    f.write("    assert logout('admin') == True\n")

print("Sample project created.")

Sample project created.


## Verifying What Was Created

In [ ]:
import os

print("Directory structure:")
print("=" * 40)
for root, dirs, files in os.walk("/content/docstring_project"):
    level = root.replace("/content/docstring_project", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = "  " * (level + 1)
    for file in files:
        print(f"{subindent}{file}")

files_to_show = [
    "/content/docstring_project/src/auth.py",
    "/content/docstring_project/src/utils.py",
    "/content/docstring_project/src/payments.py",
    "/content/docstring_project/tests/test_auth.py",
]

print("\nFile contents:")
print("=" * 40)
for filepath in files_to_show:
    print(f"\n--- {filepath} ---")
    with open(filepath, "r") as f:
        print(f.read())

Directory structure:
docstring_project/
  src/
    auth.py
    payments.py
    utils.py
  tests/
    test_auth.py

File contents:

--- /content/docstring_project/src/auth.py ---
# Authentication module

def login(user, password):
    return user == 'admin' and password == 'secret'

def logout(user):
    return True

def reset_password(user, new_password):
    return True


--- /content/docstring_project/src/utils.py ---
# Utility functions

def format_date(date):
    return date.strftime('%Y-%m-%d')

def format_currency(amount):
    return f'${amount:.2f}'

def validate_email(email):
    return '@' in email


--- /content/docstring_project/src/payments.py ---
# Payments module

def process_payment(amount, card):
    return {'status': 'success', 'amount': amount}

def refund_payment(transaction_id):
    return {'status': 'refunded', 'id': transaction_id}


--- /content/docstring_project/tests/test_auth.py ---
# Tests for auth module

def test_login():
    assert login('admin', 'secret')

## The Demo Task

All three configurations will receive the same prompt:

> *"Review the codebase and add a docstring to every function that is missing one."*

Running the same task through different permission configurations is the most direct way to see what those configurations actually do.

## The Read-Only Agent

**Configuration:**
- `allowed_tools=["Read", "Glob", "Grep"]` — Gate 3 approves these three tools
- `permission_mode="dontAsk"` — anything NOT in `allowed_tools` hits Gate 3 (denied), then Gate 4 where `dontAsk` also denies it

**Gate path:**  
Read/Glob/Grep → Gate 3 ✅  
Write/Edit → Gate 3 ❌ → Gate 4 (`dontAsk`) ❌ → **Blocked**

The agent can observe everything but touch nothing.

In [ ]:
from claude_agent_sdk import query, ClaudeAgentOptions

async for message in query(
    prompt="""
    Review the codebase at /content/docstring_project.
    Find all Python functions that are missing docstrings.
    For each function, write the docstring you would add.
    Present your findings as a clean report.
    Do NOT modify any files.
    """,
    options=ClaudeAgentOptions(
        allowed_tools=["Read", "Glob", "Grep"],
        permission_mode="dontAsk",
        model=MODEL_NAME,
    ),
):
    if hasattr(message, "result"):
        print(message.result)

Perfect! Now I have all the information I need. Let me create a comprehensive report of all functions missing docstrings.

---

# Missing Docstrings Report

## Summary
**Total Files Analyzed:** 4  
**Total Functions Found:** 13  
**Functions Missing Docstrings:** 13 (100%)

---

## Detailed Findings

### 1. **src/auth.py** - 3 functions missing docstrings

#### Function: `login`
**Line:** 3  
**Signature:** `def login(user, password):`  
**Suggested Docstring:**
```python
def login(user, password):
    """
    Authenticate a user with their credentials.
    
    Args:
        user (str): The username or user identifier.
        password (str): The user's password.
    
    Returns:
        bool: True if authentication succeeds, False otherwise.
    """
    return user == 'admin' and password == 'secret'
```

#### Function: `logout`
**Line:** 6  
**Signature:** `def logout(user):`  
**Suggested Docstring:**
```python
def logout(user):
    """
    Terminate the session for the specified 

## Verifying Files Were NOT Modified

The agent said it did not modify files. Let's not take its word for it — open `auth.py` directly and check.

In [ ]:
with open("/content/docstring_project/src/auth.py", "r") as f:
    print(f.read())

# Authentication module

def login(user, password):
    return user == 'admin' and password == 'secret'

def logout(user):
    return True

def reset_password(user, new_password):
    return True



## Auto-Edit Agent: Combo 1 — Explicit Whitelist

**Configuration:**
- `allowed_tools=["Read", "Write", "Edit", "Glob", "Grep"]` — **all five tools** pre-approved at Gate 3
- `permission_mode="dontAsk"` — mode is set, but never triggered for any of these tools

**Gate path:**  
Read/Glob/Grep → Gate 3 ✅  
Write/Edit → Gate 3 ✅ ← *same gate as reads*  
Gate 4 never reached for any tool

This is the **explicit whitelist pattern** — anyone reading the code knows exactly which tools are approved, because they are all listed.

In [ ]:
async for message in query(
    prompt="""
    Review the codebase at /content/docstring_project.
    Find all Python functions that are missing docstrings.
    Add a proper docstring to every function that is missing one.
    After making all changes, provide a summary of what you added.
    """,
    options=ClaudeAgentOptions(
        allowed_tools=["Read", "Write", "Edit", "Glob", "Grep"],
        permission_mode="dontAsk",
        model=MODEL_NAME,
    ),
):
    if hasattr(message, "result"):
        print(message.result)

## Summary

I've successfully added proper docstrings to **all 11 functions** across the codebase. Here's what was added:

### **src/auth.py** (3 functions)
1. **`login(user, password)`** - Authenticates a user with credentials, returns bool
2. **`logout(user)`** - Logs out a user, returns bool
3. **`reset_password(user, new_password)`** - Resets user password, returns bool

### **src/utils.py** (3 functions)
1. **`format_date(date)`** - Formats date objects to 'YYYY-MM-DD' string format
2. **`format_currency(amount)`** - Formats numeric amounts to currency strings with '$' prefix
3. **`validate_email(email)`** - Validates email addresses by checking for '@' symbol

### **src/payments.py** (2 functions)
1. **`process_payment(amount, card)`** - Processes payment transactions, returns status dict
2. **`refund_payment(transaction_id)`** - Refunds transactions, returns refund status dict

### **tests/test_auth.py** (2 functions)
1. **`test_login()`** - Tests login with valid admin credenti

## Verifying Files WERE Modified (Combo 1)

In [ ]:
with open("/content/docstring_project/src/auth.py", "r") as f:
    print(f.read())

# Authentication module

def login(user, password):
    """Authenticate a user with the provided credentials.

    Args:
        user (str): The username to authenticate.
        password (str): The password for authentication.

    Returns:
        bool: True if authentication is successful, False otherwise.
    """
    return user == 'admin' and password == 'secret'

def logout(user):
    """Log out a user from the system.

    Args:
        user (str): The username to log out.

    Returns:
        bool: True indicating successful logout.
    """
    return True

def reset_password(user, new_password):
    """Reset the password for a user.

    Args:
        user (str): The username whose password will be reset.
        new_password (str): The new password to set.

    Returns:
        bool: True indicating successful password reset.
    """
    return True



## Resetting Files for Combo 2

Combo 1 added docstrings to all nine functions. We need to reset the files back to their original state before running Combo 2 — otherwise the second agent will find nothing to add.

This reset is just Python writing the files again. The agent is not involved.

In [ ]:
# Reset files to original state — remove docstrings added by Combo 1
with open("/content/docstring_project/src/auth.py", "w") as f:
    f.write("# Authentication module\n\n")
    f.write("def login(user, password):\n")
    f.write("    return user == 'admin' and password == 'secret'\n\n")
    f.write("def logout(user):\n")
    f.write("    return True\n\n")
    f.write("def reset_password(user, new_password):\n")
    f.write("    return True\n")

with open("/content/docstring_project/src/utils.py", "w") as f:
    f.write("# Utility functions\n\n")
    f.write("def format_date(date):\n")
    f.write("    return date.strftime('%Y-%m-%d')\n\n")
    f.write("def format_currency(amount):\n")
    f.write("    return f'${amount:.2f}'\n\n")
    f.write("def validate_email(email):\n")
    f.write("    return '@' in email\n")

with open("/content/docstring_project/src/payments.py", "w") as f:
    f.write("# Payments module\n\n")
    f.write("def process_payment(amount, card):\n")
    f.write("    return {'status': 'success', 'amount': amount}\n\n")
    f.write("def refund_payment(transaction_id):\n")
    f.write("    return {'status': 'refunded', 'id': transaction_id}\n")

with open("/content/docstring_project/tests/test_auth.py", "w") as f:
    f.write("# Tests for auth module\n\n")
    f.write("def test_login():\n")
    f.write("    assert login('admin', 'secret') == True\n\n")
    f.write("def test_logout():\n")
    f.write("    assert logout('admin') == True\n")

print("Files reset to original state — no docstrings.")

Files reset to original state — no docstrings.


## Auto-Edit Agent: Combo 2 — Mode-Based Approval

**Configuration:**
- `allowed_tools=["Read", "Glob", "Grep"]` — only read tools pre-approved at Gate 3
- `permission_mode="acceptEdits"` — Write and Edit fall through Gate 3 to Gate 4, where `acceptEdits` approves them

**Gate path:**  
Read/Glob/Grep → Gate 3 ✅  
Write/Edit → Gate 3 ❌ → Gate 4 (`acceptEdits`) ✅

This is the **mode-based approval pattern** — the read tools are explicit, write access is delegated to the mode. The end result is **identical to Combo 1**.

In [ ]:
async for message in query(
    prompt="""
    Review the codebase at /content/docstring_project.
    Find all Python functions that are missing docstrings.
    Add a proper docstring to every function that is missing one.
    After making all changes, provide a summary of what you added.
    """,
    options=ClaudeAgentOptions(
        allowed_tools=["Read", "Glob", "Grep"],
        permission_mode="acceptEdits",
        model=MODEL_NAME,
    ),
):
    if hasattr(message, "result"):
        print(message.result)

## Summary

✅ **All Python functions now have proper docstrings!** Here's what was added:

### **src/auth.py** (3 functions)
1. **`login(user, password)`** - Authenticates a user with username and password
2. **`logout(user)`** - Logs out the specified user
3. **`reset_password(user, new_password)`** - Resets the password for a user

### **src/payments.py** (2 functions)
1. **`process_payment(amount, card)`** - Processes a payment transaction
2. **`refund_payment(transaction_id)`** - Refunds a previously processed payment

### **src/utils.py** (3 functions)
1. **`format_date(date)`** - Formats a date object as YYYY-MM-DD string
2. **`format_currency(amount)`** - Formats a numeric amount as currency with dollar sign
3. **`validate_email(email)`** - Validates email address by checking for '@' symbol

### **tests/test_auth.py** (2 functions)
1. **`test_login()`** - Tests login function with valid admin credentials
2. **`test_logout()`** - Tests logout function

**Total: 10 functions** rec

## Verifying Files WERE Modified (Combo 2)

In [ ]:
with open("/content/docstring_project/src/auth.py", "r") as f:
    print(f.read())

# Authentication module

def login(user, password):
    """
    Authenticate a user with the provided username and password.

    Args:
        user (str): The username for authentication.
        password (str): The password for authentication.

    Returns:
        bool: True if credentials match admin/secret, False otherwise.
    """
    return user == 'admin' and password == 'secret'

def logout(user):
    """
    Log out the specified user.

    Args:
        user (str): The username of the user to log out.

    Returns:
        bool: True indicating successful logout.
    """
    return True

def reset_password(user, new_password):
    """
    Reset the password for a specified user.

    Args:
        user (str): The username whose password will be reset.
        new_password (str): The new password to set.

    Returns:
        bool: True indicating successful password reset.
    """
    return True



## Three-Way Comparison

| | Read-Only Agent | Auto-Edit Combo 1 | Auto-Edit Combo 2 |
|---|---|---|---|
| `allowed_tools` | Read, Glob, Grep | Read, Write, Edit, Glob, Grep | Read, Glob, Grep |
| `permission_mode` | `dontAsk` | `dontAsk` | `acceptEdits` |
| Read tools approved at | Gate 3 | Gate 3 | Gate 3 |
| Write/Edit approved at | Never — denied | Gate 3 | Gate 4 |
| Can read files | ✅ | ✅ | ✅ |
| Can modify files | ❌ | ✅ | ✅ |
| Best for | Auditing, review, analysis | Explicit whitelist pattern | Mode-based write access |

**Key insight:** Combo 1 and Combo 2 produce **identical agent behaviour** through different gate paths.

> There is no single correct way to configure permissions. Multiple valid combinations can produce identical results. Choose the one that most clearly expresses your intent to someone reading the code.

## The Permission Spectrum — What Sits in the Middle

```
READ-ONLY                                                AUTO-EDIT
(dontAsk, read tools only)  ←— Most agents live here —→  (acceptEdits or full whitelist)
Max Safety / Zero Risk                                   Max Autonomy / Use with care
```

Most real production agents are **not** at either extreme. A common pattern:

1. **Start with a Read-Only Agent** — understand the codebase, build confidence in how the agent behaves
2. **Switch to an Auto-Edit configuration** — once you trust the output, give the agent write access for a specific, narrow operation

The right configuration depends on:
- **Trust** — how confident are you in the agent's behaviour on this task?
- **Reversibility** — can you roll back if something goes wrong? (e.g. git-tracked codebase)
- **Environment** — isolated sandbox or shared production system?
- **Task type** — analysis tasks lean left; refactoring tasks depend on your confidence level

**In Lecture 4.3** — Blocking Dangerous Operations — we learn how to intercept specific tool calls within an otherwise permissive agent and block exactly the ones that are dangerous.

## Summary — Lecture 4.2

**What we built:**
- **Read-Only Agent** — `allowed_tools=["Read", "Glob", "Grep"]`, `permission_mode="dontAsk"` — reported proposed docstrings, touched no files — verified
- **Auto-Edit Combo 1** — `allowed_tools=["Read", "Write", "Edit", "Glob", "Grep"]`, `permission_mode="dontAsk"` — added docstrings, all tools approved at Gate 3
- **Auto-Edit Combo 2** — `allowed_tools=["Read", "Glob", "Grep"]`, `permission_mode="acceptEdits"` — added identical docstrings, Write/Edit approved at Gate 4

**Key decisions:**
- `allowed_tools` always approves at Gate 3 — no exceptions
- `dontAsk` denies anything not in `allowed_tools` at Gate 4
- `acceptEdits` approves file edit tools at Gate 4
- Multiple combinations can produce identical agent behaviour — choose based on clarity of intent

**Coming up in Lecture 4.3:** Blocking Dangerous Operations — intercept and block specific tool calls within an otherwise permissive agent.